# Business Problem Discovery: 50 → 20 → 5 → 1

Notebook ini digunakan untuk:
1. membaca 50 problem statements,
2. menghitung weighted business-feasibility score,
3. meranking 50 masalah,
4. menampilkan Top 20,
5. memilih 5 MVP,
6. memasukkan hasil kampanye nyata,
7. menentukan winning product berdasarkan **ROAS**, bukan asumsi.

> Catatan: desk validation bukan product-market fit. PMF baru dapat dibahas setelah MVP memperoleh usage, conversion, retention, dan revenue signal nyata.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = [{'id': 1, 'domain': 'Sports & Lifestyle', 'problem': 'Sulit menemukan partner olahraga dengan lokasi, jadwal, dan level kemampuan yang cocok.', 'target_user': 'Mahasiswa, pekerja muda, komunitas olahraga', 'pain': 5, 'frequency': 5, 'market_size': 5, 'wtp': 3.8, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.7, 'ease_acquisition': 5, 'retention': 4.5, 'competitor': 'AYO', 'revenue_hypothesis': 'Premium matching/boost; venue referral fee'}, {'id': 2, 'domain': 'Sports & Lifestyle', 'problem': 'Sulit menemukan slot lapangan olahraga yang tersedia sekaligus membagi biaya dengan pemain lain.', 'target_user': 'Pemain badminton, futsal, tenis, padel', 'pain': 4.5, 'frequency': 4.5, 'market_size': 4.5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 3, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'AYO', 'revenue_hypothesis': 'Booking fee / venue subscription'}, {'id': 3, 'domain': 'Sports & Lifestyle', 'problem': 'Pemula sulit menemukan kelas atau komunitas olahraga yang ramah pemula dan dekat lokasi.', 'target_user': 'Pemula olahraga', 'pain': 4, 'frequency': 4, 'market_size': 4.5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3.5, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'AYO; Strava Clubs', 'revenue_hypothesis': 'Featured class / community subscription'}, {'id': 4, 'domain': 'Sports & Lifestyle', 'problem': 'Tim olahraga amatir sering kekurangan pemain pengganti saat hari pertandingan.', 'target_user': 'Tim futsal/basket/badminton amatir', 'pain': 4, 'frequency': 3.5, 'market_size': 4, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 4.5, 'retention': 3.5, 'competitor': 'AYO (indirect)', 'revenue_hypothesis': 'Match fee / premium team tools'}, {'id': 5, 'domain': 'Sports & Lifestyle', 'problem': 'Orang yang ingin rutin olahraga sulit menjaga konsistensi tanpa teman atau accountability partner.', 'target_user': 'Gen Z dan pekerja muda', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 3, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Strava Clubs; fitness apps', 'revenue_hypothesis': 'Subscription / challenge sponsorship'}, {'id': 6, 'domain': 'Housing', 'problem': 'Pencari kos sulit memastikan kondisi aktual kamar, fasilitas, dan lingkungan sesuai listing.', 'target_user': 'Mahasiswa dan pekerja perantau', 'pain': 5, 'frequency': 4, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 2.5, 'ease_acquisition': 4.5, 'retention': 3, 'competitor': 'Mamikos; Cove; Pinhome', 'revenue_hypothesis': 'Verified listing fee / premium inspection'}, {'id': 7, 'domain': 'Housing', 'problem': 'Pencari kos/kontrakan sulit menemukan roommate yang cocok dari sisi budget, kebiasaan, dan jadwal.', 'target_user': 'Mahasiswa, fresh graduate, pekerja muda', 'pain': 5, 'frequency': 4, 'market_size': 4.5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 4.2, 'ease_acquisition': 4.5, 'retention': 4.5, 'competitor': 'Roomster; grup Facebook/WhatsApp', 'revenue_hypothesis': 'Premium match; verification fee; listing boost'}, {'id': 8, 'domain': 'Housing', 'problem': 'Pencari hunian sewa sulit memverifikasi listing dan menghindari penipuan sebelum transfer deposit.', 'target_user': 'Pencari kos/kontrakan', 'pain': 5, 'frequency': 3.5, 'market_size': 4.5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 3.5, 'competition_opportunity': 3, 'ease_acquisition': 4, 'retention': 2.5, 'competitor': 'Mamikos; Pinhome', 'revenue_hypothesis': 'Verification / escrow fee'}, {'id': 9, 'domain': 'Housing', 'problem': 'Penghuni kos/kontrakan kesulitan membagi tagihan listrik, internet, belanja rumah, dan iuran secara adil.', 'target_user': 'Roommates', 'pain': 4, 'frequency': 5, 'market_size': 4.5, 'wtp': 3, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'Splitwise', 'revenue_hypothesis': 'Freemium / payment convenience fee'}, {'id': 10, 'domain': 'Housing', 'problem': 'Keluhan perbaikan di kos/kontrakan sering tercecer di chat dan tidak terlacak statusnya.', 'target_user': 'Penghuni dan pemilik kos', 'pain': 4, 'frequency': 3.5, 'market_size': 4, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 3.5, 'retention': 4, 'competitor': 'Mamikos owner tools; Maikos', 'revenue_hypothesis': 'Owner subscription'}, {'id': 11, 'domain': 'Mobility', 'problem': 'Komuter kesulitan mendapatkan informasi perjalanan multimoda yang konsisten saat ada keterlambatan atau perubahan layanan.', 'target_user': 'Komuter perkotaan', 'pain': 4.5, 'frequency': 5, 'market_size': 5, 'wtp': 3, 'digital_fit': 5, 'mvp_feasibility': 3.5, 'competition_opportunity': 2.5, 'ease_acquisition': 4, 'retention': 4, 'competitor': 'Google Maps; Moovit; operator apps', 'revenue_hypothesis': 'Ads / premium alert / B2B data'}, {'id': 12, 'domain': 'Mobility', 'problem': 'Komuter dengan rute serupa kesulitan menemukan carpool yang aman dan jadwalnya cocok.', 'target_user': 'Komuter kampus/kantor', 'pain': 4.5, 'frequency': 5, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 3, 'competition_opportunity': 3.5, 'ease_acquisition': 4, 'retention': 4.5, 'competitor': 'Ride-hailing apps (indirect)', 'revenue_hypothesis': 'Subscription / trip matching fee'}, {'id': 13, 'domain': 'Mobility', 'problem': 'Pengendara sulit mengetahui ketersediaan parkir sebelum tiba di lokasi ramai.', 'target_user': 'Pengendara kota', 'pain': 4.5, 'frequency': 4.5, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 2, 'competition_opportunity': 2.5, 'ease_acquisition': 4, 'retention': 4, 'competitor': 'Google Maps (limited); parking operators', 'revenue_hypothesis': 'Parking lead fee / B2B SaaS'}, {'id': 14, 'domain': 'Mobility', 'problem': 'Pengunjung event/konser kesulitan mencari teman berbagi perjalanan pulang-pergi untuk menekan biaya.', 'target_user': 'Pengunjung event', 'pain': 4, 'frequency': 3, 'market_size': 4.5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 4, 'ease_acquisition': 5, 'retention': 2.5, 'competitor': 'Gojek/Grab + community groups', 'revenue_hypothesis': 'Trip match fee / sponsor'}, {'id': 15, 'domain': 'Mobility', 'problem': 'Pengguna transportasi publik kesulitan membandingkan waktu, biaya, dan jumlah transit antar kombinasi moda secara sederhana.', 'target_user': 'Komuter multi-moda', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 2, 'ease_acquisition': 4.5, 'retention': 3.5, 'competitor': 'Google Maps; Moovit; operator apps', 'revenue_hypothesis': 'Ads / premium commute assistant'}, {'id': 16, 'domain': 'Education', 'problem': 'Mahasiswa kesulitan menemukan partner belajar dengan mata kuliah, jadwal, dan target akademik yang sama.', 'target_user': 'Mahasiswa', 'pain': 4, 'frequency': 4.5, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Study group apps; WhatsApp groups', 'revenue_hypothesis': 'Premium matching / campus subscription'}, {'id': 17, 'domain': 'Education', 'problem': 'Tugas kelompok sering terlambat karena pembagian tugas, deadline, dan accountability anggota tidak jelas.', 'target_user': 'Mahasiswa dan pelajar', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 2, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Notion; Trello; Google Tasks', 'revenue_hypothesis': 'Freemium / campus plan'}, {'id': 18, 'domain': 'Education', 'problem': 'Mahasiswa kesulitan mendapatkan bantuan belajar singkat untuk topik spesifik tanpa membeli paket bimbel penuh.', 'target_user': 'Mahasiswa', 'pain': 4.5, 'frequency': 4, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 3, 'ease_acquisition': 4.5, 'retention': 3.5, 'competitor': 'Ruangguru; tutor marketplaces', 'revenue_hypothesis': 'Per-session commission'}, {'id': 19, 'domain': 'Education', 'problem': 'Mahasiswa kesulitan menemukan catatan/ringkasan kuliah yang relevan dan dapat dipercaya untuk mata kuliah tertentu.', 'target_user': 'Mahasiswa', 'pain': 4, 'frequency': 4, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 3.5, 'ease_acquisition': 5, 'retention': 3, 'competitor': 'Studocu; class groups', 'revenue_hypothesis': 'Subscription / creator revenue share'}, {'id': 20, 'domain': 'Education', 'problem': 'Mahasiswa kesulitan menemukan proyek nyata atau tim lintas jurusan untuk membangun portofolio.', 'target_user': 'Mahasiswa tingkat akhir', 'pain': 4.5, 'frequency': 4, 'market_size': 5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 4, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'LinkedIn; Glints communities (indirect)', 'revenue_hypothesis': 'Recruiter fee / premium project tools'}, {'id': 21, 'domain': 'Career', 'problem': 'Fresh graduate kesulitan mengetahui gap CV mereka terhadap lowongan tertentu.', 'target_user': 'Fresh graduate', 'pain': 4.5, 'frequency': 5, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 5, 'retention': 3.5, 'competitor': 'Glints; KitaLulus; AI CV tools', 'revenue_hypothesis': 'CV review subscription / one-off fee'}, {'id': 22, 'domain': 'Career', 'problem': 'Mahasiswa dan pencari kerja sulit menemukan part-time atau gig dekat lokasi yang benar-benar terverifikasi.', 'target_user': 'Mahasiswa, lulusan SMA/SMK, fresh graduate', 'pain': 5, 'frequency': 5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 2, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Glints; KitaLulus', 'revenue_hypothesis': 'Employer posting fee / success fee'}, {'id': 23, 'domain': 'Career', 'problem': 'Pelamar sulit melacak status banyak lamaran kerja dan follow-up yang harus dilakukan.', 'target_user': 'Active job seekers', 'pain': 3.5, 'frequency': 5, 'market_size': 5, 'wtp': 2, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3, 'ease_acquisition': 5, 'retention': 3, 'competitor': 'Notion; spreadsheets; job apps', 'revenue_hypothesis': 'Freemium'}, {'id': 24, 'domain': 'Career', 'problem': 'Freelancer pemula kesulitan mengelola scope, milestone, invoice, dan bukti pekerjaan dalam satu alur.', 'target_user': 'Freelancer pemula', 'pain': 4, 'frequency': 4.5, 'market_size': 4.5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3, 'ease_acquisition': 4, 'retention': 4.5, 'competitor': 'Upwork/Fiverr tools; invoice apps', 'revenue_hypothesis': 'Subscription / payment fee'}, {'id': 25, 'domain': 'Career', 'problem': 'Pencari kerja kesulitan latihan interview yang spesifik terhadap posisi dan mendapatkan feedback cepat.', 'target_user': 'Fresh graduate', 'pain': 4, 'frequency': 4.5, 'market_size': 5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 5, 'retention': 3.5, 'competitor': 'ChatGPT; interview AI tools', 'revenue_hypothesis': 'Pay-per-session / subscription'}, {'id': 26, 'domain': 'Personal Finance', 'problem': 'Pengguna sering lupa langganan digital dan tanggal renewal sehingga pengeluaran berulang sulit dikendalikan.', 'target_user': 'Pengguna streaming/SaaS/AI tools', 'pain': 4, 'frequency': 4.5, 'market_size': 5, 'wtp': 3, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Digcos; Subsilo; other trackers', 'revenue_hypothesis': 'Freemium / premium reminder'}, {'id': 27, 'domain': 'Personal Finance', 'problem': 'Teman serumah/rombongan kesulitan membagi pengeluaran dan menagih pembayaran secara nyaman.', 'target_user': 'Roommates, traveler groups', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Splitwise', 'revenue_hypothesis': 'Premium / payment fee'}, {'id': 28, 'domain': 'Personal Finance', 'problem': 'Anak muda sulit mengubah transaksi harian menjadi gambaran sederhana tentang pola pengeluaran dan target tabungan.', 'target_user': 'Gen Z dan pekerja muda', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 2, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Budgeting apps', 'revenue_hypothesis': 'Freemium / premium insights'}, {'id': 29, 'domain': 'Personal Finance', 'problem': 'Keluarga muda kesulitan mengelola pengingat tagihan rutin dan siapa yang bertanggung jawab membayarnya.', 'target_user': 'Keluarga muda/roommates', 'pain': 3.5, 'frequency': 5, 'market_size': 4.5, 'wtp': 3, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3.5, 'ease_acquisition': 4, 'retention': 4, 'competitor': 'Google Calendar; Todoist', 'revenue_hypothesis': 'Family plan'}, {'id': 30, 'domain': 'Personal Finance', 'problem': 'Konsumen sulit menyimpan dan menemukan kembali garansi, struk, serta masa berlaku produk yang dibeli.', 'target_user': 'Konsumen rumah tangga', 'pain': 3.5, 'frequency': 3, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 4, 'retention': 2.5, 'competitor': 'Cloud storage / wallet apps (indirect)', 'revenue_hypothesis': 'Freemium / retailer B2B'}, {'id': 31, 'domain': 'Food & Commerce', 'problem': 'Makanan layak konsumsi yang tidak terjual di restoran/kafe sulit dipertemukan cepat dengan pembeli diskon di sekitar.', 'target_user': 'Mahasiswa, pekerja, restoran/kafe', 'pain': 5, 'frequency': 4.5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 3.5, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Surplus Indonesia', 'revenue_hypothesis': '10–15% transaction commission'}, {'id': 32, 'domain': 'Food & Commerce', 'problem': 'Mahasiswa/pekerja sulit menemukan paket makan harian sesuai budget, lokasi, dan kebutuhan diet tanpa mencari manual satu per satu.', 'target_user': 'Mahasiswa dan pekerja', 'pain': 4.5, 'frequency': 5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 3.2, 'ease_acquisition': 5, 'retention': 4.5, 'competitor': 'GoFood/GrabFood; catering sellers', 'revenue_hypothesis': 'Merchant commission / subscription'}, {'id': 33, 'domain': 'Food & Commerce', 'problem': 'Antrean kantin atau tenant makan membuat waktu istirahat terbuang karena tidak ada pre-order pickup yang sederhana.', 'target_user': 'Mahasiswa/karyawan kampus dan kantor', 'pain': 4, 'frequency': 5, 'market_size': 4.5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Food ordering apps (indirect)', 'revenue_hypothesis': 'Merchant SaaS / per-order fee'}, {'id': 34, 'domain': 'Food & Commerce', 'problem': 'Pesanan makan kelompok sulit dikonsolidasikan dari pilihan menu sampai pembagian pembayaran.', 'target_user': 'Kelompok teman/kantor', 'pain': 3.5, 'frequency': 4, 'market_size': 4.5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3, 'ease_acquisition': 5, 'retention': 3, 'competitor': 'Splitwise + food delivery apps', 'revenue_hypothesis': 'Affiliate / premium group order'}, {'id': 35, 'domain': 'Food & Commerce', 'problem': 'Konsumen sulit menemukan penjual makanan rumahan terpercaya di radius dekat dengan jadwal menu yang jelas.', 'target_user': 'Konsumen lokal', 'pain': 4, 'frequency': 4, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 3.5, 'competition_opportunity': 3.5, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'GoFood/GrabFood; local communities', 'revenue_hypothesis': 'Commission / listing fee'}, {'id': 36, 'domain': 'Local Services', 'problem': 'Konsumen sulit mengetahui penyedia jasa rumah tangga yang benar-benar tersedia pada waktu yang dibutuhkan.', 'target_user': 'Rumah tangga perkotaan', 'pain': 4.5, 'frequency': 4, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 2.5, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'Pinhome Home Service', 'revenue_hypothesis': 'Commission per booking'}, {'id': 37, 'domain': 'Local Services', 'problem': 'Salon, barbershop, dan jasa berbasis appointment menghadapi double booking dan no-show karena booking masih lewat chat/manual.', 'target_user': 'Salon, barbershop, nail studio, beauty service', 'pain': 5, 'frequency': 5, 'market_size': 4.5, 'wtp': 5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 4, 'retention': 5, 'competitor': 'Kasera; Fresha', 'revenue_hypothesis': 'SaaS Rp49k–99k/bulan + deposit fee'}, {'id': 38, 'domain': 'Local Services', 'problem': 'Pelanggan laundry kesulitan mengetahui status cucian dan waktu selesai tanpa bertanya melalui chat.', 'target_user': 'Laundry kiloan', 'pain': 3.5, 'frequency': 4.5, 'market_size': 4.5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4, 'ease_acquisition': 4, 'retention': 4.5, 'competitor': 'Laundry POS/status apps', 'revenue_hypothesis': 'SaaS merchant'}, {'id': 39, 'domain': 'Local Services', 'problem': 'Pemilik hewan kesulitan menemukan pet sitter/boarding yang tersedia, dekat, dan memiliki rekam reputasi.', 'target_user': 'Pet owners', 'pain': 4.5, 'frequency': 3.5, 'market_size': 4.5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 3.5, 'competition_opportunity': 3.5, 'ease_acquisition': 4, 'retention': 3.5, 'competitor': 'Pet boarding marketplaces', 'revenue_hypothesis': 'Commission per booking'}, {'id': 40, 'domain': 'Local Services', 'problem': 'Orang yang hanya sesekali membutuhkan kamera, alat camping, alat bayi, atau tools sulit menemukan penyewaan tepercaya di dekatnya.', 'target_user': 'Mahasiswa, creator, keluarga muda', 'pain': 4, 'frequency': 3, 'market_size': 4.5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 4, 'competition_opportunity': 4, 'ease_acquisition': 4.5, 'retention': 3.5, 'competitor': 'Grover (global); local rental shops', 'revenue_hypothesis': 'Transaction fee / insurance fee'}, {'id': 41, 'domain': 'Community', 'problem': 'Barang hilang di kampus, stasiun, atau lingkungan sering sulit dipertemukan dengan penemu karena informasi tersebar.', 'target_user': 'Mahasiswa, komuter, masyarakat umum', 'pain': 4, 'frequency': 3, 'market_size': 5, 'wtp': 2, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3.5, 'ease_acquisition': 4.5, 'retention': 2, 'competitor': 'BantuCari; NEMU', 'revenue_hypothesis': 'Ads / institution subscription'}, {'id': 42, 'domain': 'Community', 'problem': 'Orang yang pindah kota atau memiliki hobi baru sulit menemukan komunitas lokal yang aktif dan sesuai minat.', 'target_user': 'Mahasiswa/pekerja baru', 'pain': 4, 'frequency': 4, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 3, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Meetup; Facebook Groups; Strava Clubs', 'revenue_hypothesis': 'Premium discovery / event fee'}, {'id': 43, 'domain': 'Community', 'problem': 'Calon relawan kesulitan menemukan kegiatan volunteer lokal yang sesuai waktu, lokasi, dan kemampuan.', 'target_user': 'Mahasiswa/pekerja muda', 'pain': 3.5, 'frequency': 3, 'market_size': 4, 'wtp': 1.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 4.5, 'ease_acquisition': 4, 'retention': 2, 'competitor': 'Volunteer platforms / social media', 'revenue_hypothesis': 'Organization fee / sponsorship'}, {'id': 44, 'domain': 'Sustainability', 'problem': 'Warga kesulitan mengetahui lokasi bank sampah, jenis sampah yang diterima, harga, dan jadwal penyerahan/pickup.', 'target_user': 'Rumah tangga perkotaan', 'pain': 4, 'frequency': 4, 'market_size': 5, 'wtp': 2.5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 4, 'ease_acquisition': 4, 'retention': 3.5, 'competitor': 'Bank sampah apps', 'revenue_hypothesis': 'B2B municipality / pickup fee'}, {'id': 45, 'domain': 'Sharing Economy', 'problem': 'Warga/mahasiswa memiliki barang jarang dipakai tetapi tidak memiliki cara aman untuk meminjamkan atau menyewakannya ke lingkungan sekitar.', 'target_user': 'Mahasiswa, komunitas hunian', 'pain': 4, 'frequency': 3.5, 'market_size': 5, 'wtp': 4, 'digital_fit': 5, 'mvp_feasibility': 3.5, 'competition_opportunity': 4, 'ease_acquisition': 4.5, 'retention': 4, 'competitor': 'Grover (global); Facebook Marketplace (indirect)', 'revenue_hypothesis': 'Transaction fee / protection fee'}, {'id': 46, 'domain': 'UMKM', 'problem': 'Seller menerima order dari WhatsApp, Instagram, marketplace, dan offline sehingga pesanan mudah tercecer.', 'target_user': 'UMKM social commerce', 'pain': 5, 'frequency': 5, 'market_size': 5, 'wtp': 5, 'digital_fit': 5, 'mvp_feasibility': 4.5, 'competition_opportunity': 3, 'ease_acquisition': 4.5, 'retention': 5, 'competitor': 'BukuWarung/POS tools; marketplace seller centers', 'revenue_hypothesis': 'SaaS Rp49k–99k/bulan'}, {'id': 47, 'domain': 'UMKM', 'problem': 'UMKM retail/F&B sering terlambat restock karena catatan stok dan batas minimum tidak terkelola.', 'target_user': 'UMKM retail/F&B', 'pain': 5, 'frequency': 5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 1.5, 'ease_acquisition': 4, 'retention': 5, 'competitor': 'Majoo/Moka/Olsera/POS apps', 'revenue_hypothesis': 'SaaS'}, {'id': 48, 'domain': 'UMKM', 'problem': 'Pemilik usaha mikro sulit mengetahui kas dan laba karena pencatatan pemasukan-pengeluaran tidak konsisten.', 'target_user': 'UMKM mikro', 'pain': 5, 'frequency': 5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 1, 'ease_acquisition': 4.5, 'retention': 5, 'competitor': 'BukuWarung; SIAPIK; bookkeeping apps', 'revenue_hypothesis': 'Freemium / SaaS'}, {'id': 49, 'domain': 'UMKM', 'problem': 'UMKM tidak memiliki sistem sederhana untuk mengingat pelanggan lama dan mendorong repeat purchase.', 'target_user': 'UMKM retail/F&B/jasa', 'pain': 4.5, 'frequency': 4.5, 'market_size': 5, 'wtp': 4.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 3.5, 'retention': 5, 'competitor': 'CRM/POS loyalty tools', 'revenue_hypothesis': 'SaaS'}, {'id': 50, 'domain': 'UMKM', 'problem': 'Pemilik UMKM kesulitan menjaga kalender konten dan menilai konten mana yang menghasilkan leads/penjualan.', 'target_user': 'UMKM aktif di media sosial', 'pain': 4, 'frequency': 5, 'market_size': 5, 'wtp': 3.5, 'digital_fit': 5, 'mvp_feasibility': 5, 'competition_opportunity': 2.5, 'ease_acquisition': 5, 'retention': 4, 'competitor': 'Canva; Meta Business Suite; Buffer', 'revenue_hypothesis': 'Subscription'}]
df = pd.DataFrame(data)

weights = {
    "pain": 0.15,
    "frequency": 0.10,
    "market_size": 0.15,
    "wtp": 0.15,
    "digital_fit": 0.10,
    "mvp_feasibility": 0.10,
    "competition_opportunity": 0.10,
    "ease_acquisition": 0.10,
    "retention": 0.05
}

df["business_score"] = sum(df[col] * w for col, w in weights.items()) * 20
df = df.sort_values("business_score", ascending=False).reset_index(drop=True)
df.insert(0,"rank",range(1,len(df)+1))
df.head(20)


In [ ]:

plt.figure(figsize=(11,8))
top20 = df.head(20).sort_values("business_score")
plt.barh(top20["problem"].str.slice(0,60), top20["business_score"])
plt.xlabel("Business feasibility score")
plt.title("Top 20 Business Problems")
plt.tight_layout()
plt.show()


## Current Top 5 MVP Candidates

1. **Slotly**
2. **OrderNest**
3. **SisaRasa**
4. **TemanMain Campus**
5. **RoomieFit**

In [ ]:

top5_ids = [37, 46, 31, 1, 7]
top5 = df[df["id"].isin(top5_ids)].sort_values("business_score", ascending=False)
top5[["rank","id","domain","problem","business_score","competitor","revenue_hypothesis"]]


## Real Marketing Experiment

Isi angka **aktual** setelah 5 MVP dipasarkan. Jangan mengisi dengan estimasi jika tugas meminta bukti kampanye nyata.

ROAS = Attributed Revenue / Ad Spend

Winning product sebaiknya hanya dipilih dari produk yang memiliki minimum evidence, misalnya minimal 3 paid customers, agar satu transaksi kebetulan tidak langsung dianggap product-market fit.


In [ ]:

campaign = pd.DataFrame({
    "product": ['Slotly', 'OrderNest', 'SisaRasa', 'TemanMain Campus', 'RoomieFit'],
    "impressions": [0]*5,
    "clicks": [0]*5,
    "landing_visitors": [0]*5,
    "signups": [0]*5,
    "activated_users": [0]*5,
    "paid_customers": [0]*5,
    "ad_spend": [0]*5,
    "attributed_revenue": [0]*5
})
campaign


In [ ]:

def evaluate_campaign(c):
    c = c.copy()
    c["ctr"] = np.where(c["impressions"]>0, c["clicks"]/c["impressions"], 0)
    c["landing_cvr"] = np.where(c["landing_visitors"]>0, c["signups"]/c["landing_visitors"], 0)
    c["cac"] = np.where(c["paid_customers"]>0, c["ad_spend"]/c["paid_customers"], np.nan)
    c["roas"] = np.where(c["ad_spend"]>0, c["attributed_revenue"]/c["ad_spend"], 0)
    c["eligible"] = c["paid_customers"] >= 3
    return c

results = evaluate_campaign(campaign)
results


In [ ]:

eligible = results[results["eligible"]].copy()
if len(eligible) == 0:
    print("Belum ada winning product yang valid: tidak ada produk dengan minimum 3 paid customers.")
else:
    winner = eligible.sort_values("roas", ascending=False).iloc[0]
    print("Winning product:", winner["product"])
    print("ROAS:", round(winner["roas"], 2), "x")
